[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/evals_exploration.ipynb)

# MarinFold evals exploration — [#250](https://github.com/Open-Athena/MarinFold/issues/250)

contacts-v1 predicts a protein's **residue–residue contact map from its sequence alone**, by
generating a document of `<contact> <pI> <pJ>` statements. Our published numbers for that are
aggregates — a mean R-precision over an eval set. This notebook is the per-protein view underneath
them.

Five parts, each usable on its own:

| | what it does | needs |
|---|---|---|
| **1. Scoreboard** | R-precision / AUC for every predictor on a chosen eval set, with bootstrap CIs | CPU |
| **2. Protein browser** | every protein × every predictor, joined to structural / homology / annotation features, and a paired contrast between any two of them | CPU |
| **3. Contact maps** | run a checkpoint on one protein and plot its prediction against ground truth | **GPU** |
| **4. Two checkpoints** | the same protein under two models, side by side | **GPU** |
| **5. Manuscript figures** | publication-ready panels written to `figures/`: contact-prediction R-precision and Helico GDT-TS for both protein classes, and a contact map for Top7 | CPU (+ GPU for the map) |

Everything is read from the public `open-athena/MarinFold` bucket — **no token, no cluster**.
Parts 3–4 need a GPU runtime (*Runtime → Change runtime type → T4*); the model is a 1.5B
checkpoint downloaded from the bucket.

### The two eval universes

They are kept separate on purpose and **must not be pooled or compared across**:

* **`foldbench-monomers`** — [#245](https://github.com/Open-Athena/MarinFold/issues/245)'s 333 FoldBench
  monomers, cut into `eval-val` (97 natural, the working set), `eval-test` (217 natural, held out —
  see the read budget below) and `eval-denovo` (19 de novo designs).
* **`legacy-554`** — the historical 554-protein set every published MarinFold number before #245
  lives on. 75 % de novo designed, and selected on for a year, so it answers "how does this compare
  to our earlier checkpoints" and nothing about generalisation.

They overlap in 112 stems but **disagree on the input sequence for 11 of them** (different chain
resolution), so a stem scored under one universe is not comparable to the same stem under the other.

In [ ]:
# @title Configuration { run: "auto", display-mode: "form" }

# --- what to look at -----------------------------------------------------------------
UNIVERSE = "foldbench-monomers"  # @param ["foldbench-monomers", "legacy-554"]
EVAL_SET = "eval-val"  # @param ["eval-val", "eval-test", "eval-denovo", "FoldBench natural monomers", "everything"]
RANGE = "all"  # @param ["all", "long", "medium", "short"]
METRIC = "R"  # @param ["R", "AUC", "L", "L/2", "L/5"]

# --- which protein to fold (parts 3-4) -----------------------------------------------
# Any stem in the selected universe. Part 2 prints the candidates; `1qys_A` (Top7, a de novo
# design) and `1ubq_A` live in legacy-554, not in the FoldBench monomer sets.
PROTEIN = "8ah9_A"  # @param {type:"string"}

MODEL = "contacts-v1-exp199-cooldown-1.5B"  # @param ["contacts-v1-exp199-cooldown-1.5B", "contacts-v1-exp232-m2-p06-1.5B", "contacts-v1-exp199-1.5B", "contacts-v1-exp166-1.5B", "contacts-v1-exp117-1.5B", "contacts-v1-exp75-1.5B"]
MODEL_B = "contacts-v1-exp232-m2-p06-1.5B"  # @param ["contacts-v1-exp232-m2-p06-1.5B", "contacts-v1-exp117-1.5B", "contacts-v1-exp75-1.5B", "contacts-v1-exp166-1.5B", "contacts-v1-exp199-1.5B", "contacts-v1-exp199-cooldown-1.5B"]

# --- inference recipe (exp82 settled values; change only deliberately) ----------------
N_ROLLOUTS = 100  # @param {type:"integer"}
TEMPERATURE = 1.0  # @param {type:"number"}
TOP_P = 0.95  # @param {type:"number"}
TOP_K = -1  # @param {type:"integer"}
BATCH_SIZE = 0  # @param {type:"integer"}
# 0 = size the rollout batch from free GPU memory, up to N_ROLLOUTS (all rollouts for one
# protein decoded in a single pass, which is what an A100 has room for). Set a positive number
# to force one.
# "auto" picks bfloat16 on compute capability >= 8.0 and float16 below it (Colab's T4 is 7.5,
# where bfloat16 has no tensor cores and runs emulated). Force one if you suspect trouble.
DTYPE = "auto"  # @param ["auto", "bfloat16", "float16", "float32"]
# "auto" uses vLLM on Ampere or newer (compute capability >= 8.0 — Colab Pro's L4/A100) and
# transformers elsewhere. vLLM is several times faster on a rollout and is what the published
# eval harness ran, so it also removes one of this notebook's two divergences from it.
BACKEND = "auto"  # @param ["auto", "transformers", "vllm"]
# vLLM only. Each fold builds its own engine and vLLM does not release one when it goes out of
# scope, so a session that runs parts 3, 4 and 5 holds three at once — hence a share that fits
# three, not the 0.85 a single run would want. Raise it if you are only folding once, and
# restart the runtime if you hit "No available memory for the cache blocks".
GPU_MEMORY_UTILIZATION = 0.28  # @param {type:"number"}

# --- which published predictors parts 1-2 centre on ------------------------------------
# FOCUS is the predictor the browser sorts by and the scoreboard highlights; BASELINE is what
# it is differenced against. The #232 names exist only in `foldbench-monomers` and the
# `MarinFold #…` names only in `legacy-554`; part 2 says so and lists the alternatives if the
# pair you pick was not scored on the universe you picked.
FOCUS = "#232 m2-p06 (decontaminated)"  # @param ["#232 m2-p06 (decontaminated)", "#232 m1-p02 (decontaminated)", "#199 cooldown (contaminated)", "MarinFold #199 cooldown", "MarinFold #75", "ESMFold2", "ESMFold", "Protenix-v2 single-seq", "Protenix-v2 + MSA", "seq-KNN (decontaminated corpus)", "seq-KNN (unfiltered corpus)"]
BASELINE = "ESMFold2"  # @param ["ESMFold2", "ESMFold", "Protenix-v2 single-seq", "Protenix-v2 + MSA", "seq-KNN (unfiltered corpus)", "seq-KNN (decontaminated corpus)", "#199 cooldown (contaminated)"]

# --- part 5, the manuscript figures ----------------------------------------------------
# The protein whose predicted contact map gets drawn, and the checkpoint that predicts it.
# 1qys_A (Top7) lives in legacy-554, not in the FoldBench monomer sets, so part 5 loads that
# universe itself regardless of UNIVERSE above.
FIGURE_PROTEIN = "denovo_pdb__1qys_A"  # @param {type:"string"}
FIGURE_MODEL = "contacts-v1-exp232-m2-p06-1.5B"  # @param ["contacts-v1-exp232-m2-p06-1.5B", "contacts-v1-exp199-cooldown-1.5B", "contacts-v1-exp199-1.5B", "contacts-v1-exp117-1.5B", "contacts-v1-exp75-1.5B"]
print({k: v for k, v in globals().items() if k.isupper() and not k.startswith("_")})

In [ ]:
# @title Install (≈1 min, or ≈4 with vLLM) { display-mode: "form" }
# marinfold pins transformers<5 (the Levanter rope config is silently misread by 4.x otherwise
# — see MODELS.yaml). That pin also holds huggingface_hub < 1.0, so bucket files are read here
# over plain HTTPS resolve URLs rather than the hf_hub bucket API.
import importlib, importlib.util, shutil, subprocess, sys
from pathlib import Path


def repo_root(start: Path):
    """The MarinFold checkout containing `start`, if we are already inside one."""
    for candidate in (start, *start.parents):
        if (candidate / "marinfold" / "pyproject.toml").exists():
            return candidate
    return None


def compute_capability():
    """Highest NVIDIA compute capability present (7.5 for a T4, 8.0 for an A100), or None."""
    if shutil.which("nvidia-smi") is None:
        return None
    result = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                            capture_output=True, text=True, check=False)
    caps = [float(line) for line in result.stdout.split() if line.replace(".", "").isdigit()]
    return max(caps) if caps else None


# vLLM's wheels need Ampere or newer, so a T4 stays on transformers whatever BACKEND says.
CAPABILITY = compute_capability()
WANT_VLLM = BACKEND in ("auto", "vllm") and CAPABILITY is not None and CAPABILITY >= 8.0
if BACKEND == "vllm" and not WANT_VLLM:
    print(f"note: BACKEND='vllm' but this GPU is compute capability {CAPABILITY} "
          f"(vLLM needs >= 8.0) — falling back to transformers")

# The notebook and the checkout are two different things: opening this file from a branch link
# still clones `main` below, and a checkpoint registered on that branch is then missing from
# MODELS.yaml — `KeyError: Model '...' is not listed in MODELS.yaml`. So after cloning we check
# that every nickname the dropdowns offer actually resolves, and fall back to the branch if not.
REPO_URL = "https://github.com/Open-Athena/MarinFold.git"
REPO_FALLBACK_REF = "exp250/evals-exploration-notebook"
REQUIRED_MODELS = sorted({MODEL, MODEL_B, FIGURE_MODEL})

REPO = (Path("/content/MarinFold") if Path("/content").exists()
        else repo_root(Path.cwd()) or Path.cwd() / "MarinFold")
if not (REPO / "marinfold" / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)


def registered_models() -> set:
    """Nicknames in the checkout's MODELS.yaml, read as text (marinfold may not be installed)."""
    path = REPO / "marinfold" / "marinfold" / "MODELS.yaml"
    return {line.split("nickname:", 1)[1].strip()
            for line in path.read_text().splitlines() if "nickname:" in line}


missing = [name for name in REQUIRED_MODELS if name not in registered_models()]
if missing:
    print(f"{missing} are not in this checkout's MODELS.yaml — fetching {REPO_FALLBACK_REF}")
    subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_FALLBACK_REF],
                   cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "FETCH_HEAD"], cwd=REPO, check=True)
    still_missing = [name for name in REQUIRED_MODELS if name not in registered_models()]
    if still_missing:
        raise SystemExit(f"{still_missing} are not registered on {REPO_FALLBACK_REF} either — "
                         f"check the nickname, or point MODEL at a local checkpoint directory")

install = importlib.util.find_spec("marinfold") is None
install_vllm = WANT_VLLM and importlib.util.find_spec("vllm") is None
if install or install_vllm:
    # Colab's driver is CUDA 12.x but vLLM's default PyPI wheel is built for CUDA 13, which
    # fails at import with `libcudart.so.13`. Pull the pinned cu129 wheel from vLLM's own index
    # instead. marinfold and vLLM are installed in ONE command so pip resolves transformers once
    # against both constraints (marinfold's <5 and vLLM's) rather than fighting over it.
    arguments = ["-e", f"{REPO / 'marinfold'}[transformers]"]
    if install_vllm:
        version = "0.20.2"
        arguments += [f"vllm=={version}+cu129",
                      "--extra-index-url", f"https://wheels.vllm.ai/{version}/cu129/",
                      "--extra-index-url", "https://download.pytorch.org/whl/cu129"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *arguments], check=True)
    # An editable install works by dropping a .pth file into site-packages, and .pth files are
    # only read when the interpreter starts — so the kernel that just ran pip still cannot
    # import it. Point sys.path at the package directory instead of making anyone restart the
    # runtime and re-run everything.
    if str(REPO / "marinfold") not in sys.path:
        sys.path.insert(0, str(REPO / "marinfold"))
    importlib.invalidate_caches()
if importlib.util.find_spec("marinfold") is None:
    raise SystemExit(f"marinfold is still not importable from {REPO / 'marinfold'} — "
                     f"restart the runtime (Runtime -> Restart session) and run this cell again")

HAVE_VLLM = WANT_VLLM and importlib.util.find_spec("vllm") is not None
print(f"repo: {REPO} · marinfold importable · GPU compute capability {CAPABILITY} · "
      f"vLLM {'available' if HAVE_VLLM else 'not used'}")

In [ ]:
# @title Load the published evals { display-mode: "form" }
import io, json, urllib.request
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BUCKET = "https://huggingface.co/buckets/open-athena/MarinFold/resolve"
EXP245 = f"{BUCKET}/data/contacts-v1-foldbench-monomers-exp245"
EXP89 = f"{BUCKET}/data/contacts-v1-model-eval-exp89"
EXP247 = f"{BUCKET}/data/contacts-v1-protein-properties-exp247"
EXP199_COOLDOWN_ROWS = (
    f"{BUCKET}/data/contacts-v1-model-eval-exp199/replicates/cooldown-v2-20260815-01/derived/"
    "prot-exp199-cw-cv1-p06-cool-s01/step-290400/contact_eval_cw_p06_cool_step290400_rows.csv.gz"
)
# The legacy table names predictors by (model, mode); these are the rows to keep and what to call
# them. `distogram` rows are Protenix read out a second way and would double-count the predictor.
LEGACY_PREDICTORS = {
    ("marinfold-contacts-v1", "single_seq", "lm"): "MarinFold #75",
    ("esmfold", "single_seq", "structure"): "ESMFold",
    ("esmfold2", "single_seq", "structure"): "ESMFold2",
    ("protenix-v2", "single_seq", "structure"): "Protenix-v2 single-seq",
    ("protenix-v2", "msa", "structure"): "Protenix-v2 + MSA",
}


def fetch(url: str) -> bytes:
    """Read one public bucket file into memory (anonymous)."""
    with urllib.request.urlopen(url) as response:
        return response.read()


def load_universe(name: str) -> tuple[pd.DataFrame, dict]:
    """`(targets, ground_truth)` for one eval universe.

    `targets` is one row per scorable unit: dataset, stem, L, the exact input sequence the model
    is prompted with, and whatever annotation that universe carries. `ground_truth` maps
    `(dataset, stem)` to #89's record — resolved residues and pyconfind contacts with degrees.
    """
    if name == "foldbench-monomers":
        targets = pq.read_table(io.BytesIO(
            fetch(f"{EXP245}/eval_targets_foldbench_monomers.parquet"))).to_pandas()
        annotation = pd.read_csv(io.BytesIO(fetch(f"{EXP245}/eval_sets.csv")))
        targets = targets.merge(
            annotation[["stem", "eval_set", "designed", "is_viral", "kingdom", "title",
                        "deposit_date", "exp199_best_identity", "exp199_stratum"]],
            on="stem", how="left", validate="one_to_one")
        ground_truth_bytes = fetch(f"{EXP245}/gt_universe_scored.jsonl")
    elif name == "legacy-554":
        # The 554 input sequences are not on the bucket as a table; exp94's query FASTA is the
        # same set, verified byte-identical against the prompts exp89 actually used (554/554).
        rows = []
        fasta = (REPO / "experiments/exp94_evals_sequence_knn_baseline/data/eval_queries.fasta")
        for line in fasta.read_text().splitlines():
            if line.startswith(">"):
                dataset, stem = line[1:].strip().split("__", 1)
                rows.append({"dataset": dataset, "stem": stem, "input_seq": ""})
            elif line.strip():
                rows[-1]["input_seq"] += line.strip()
        targets = pd.DataFrame(rows)
        targets["L"] = targets.input_seq.str.len()
        targets["eval_set"] = "legacy-554"
        targets["designed"] = (targets.dataset == "denovo_pdb").astype(int)
        ground_truth_bytes = fetch(f"{EXP89}/gt_universe.jsonl")
    else:
        raise ValueError(f"unknown universe {name!r}")

    ground_truth = {}
    for line in ground_truth_bytes.decode().splitlines():
        record = json.loads(line)
        ground_truth[(record["dataset"], record["stem"])] = record
    missing = set(zip(targets.dataset, targets.stem)) - set(ground_truth)
    if missing:
        raise ValueError(f"{len(missing)} targets have no ground truth, e.g. {sorted(missing)[:3]}")
    return targets, ground_truth


def load_published(name: str) -> pd.DataFrame:
    """Published per-protein scores: `dataset, stem, predictor, range, cut, value`.

    Keyed by (dataset, stem), not stem: `7ur7_A` and `8ah9_A` are each in legacy-554 twice, under
    two datasets, with *different* input sequences and lengths. Joining on stem alone silently
    duplicates them and averages two different proteins together.
    """
    columns = ["dataset", "stem", "predictor", "range", "cut", "value"]
    if name == "foldbench-monomers":
        scores = pd.read_csv(io.BytesIO(fetch(f"{EXP245}/per_protein.csv.gz")), compression="gzip")
        # #245 scored one dataset only, and dropped the duplicate stems, so the key is safe to add.
        return scores.rename(columns={"precision": "value"}).assign(dataset="foldbench_monomer")[columns]

    legacy = pd.read_csv(io.BytesIO(fetch(f"{EXP89}/contact_precision_all.csv")))
    cooldown = pd.read_csv(io.BytesIO(fetch(EXP199_COOLDOWN_ROWS)), compression="gzip")
    cooldown = cooldown.assign(model="marinfold-contacts-v1", mode="single_seq")  # relabel below
    frames = []
    for frame, mapping in ((legacy, LEGACY_PREDICTORS),
                           (cooldown, {("marinfold-contacts-v1", "single_seq", "lm"):
                                       "MarinFold #199 cooldown"})):
        keys = list(zip(frame.model, frame["mode"], frame.predictor))
        frame = frame.assign(label=[mapping.get(key) for key in keys])
        frames.append(frame[frame.label.notna()])
    scores = pd.concat(frames, ignore_index=True)
    return (scores.rename(columns={"label": "predictor_name", "precision": "value"})
                  .drop(columns=["predictor"]).rename(columns={"predictor_name": "predictor"})
                  [columns])


def load_features() -> pd.DataFrame:
    """#247's 75 per-protein features (FoldBench monomers only)."""
    return pd.read_csv(io.BytesIO(fetch(f"{EXP247}/protein_features.csv")))


targets, ground_truth = load_universe(UNIVERSE)
published = load_published(UNIVERSE)
features = load_features()
# Some published tables cover a superset of the universe — the #199 cooldown was also scored on
# #226's 23 extra FoldBench chains, which are not part of the 554. Keep only this universe's units
# so a mean is over exactly the set it claims to be over.
in_universe = set(zip(targets.dataset, targets.stem))
outside = [key not in in_universe for key in zip(published.dataset, published.stem)]
if any(outside):
    dropped = published[outside]
    print(f"note: dropped {len(set(zip(dropped.dataset, dropped.stem)))} scored units that are not "
          f"in {UNIVERSE}")
    published = published[[not flag for flag in outside]]
print(f"{UNIVERSE}: {len(targets)} units, {published.predictor.nunique()} published predictors")
print(targets.eval_set.value_counts().to_string())

## 1. The scoreboard

Mean over proteins of the per-protein metric, with a percentile bootstrap CI over proteins.
Three rules from [#245](https://github.com/Open-Athena/MarinFold/issues/245) are enforced in the
output rather than left to the reader, because ignoring them changes the conclusion:

* **Designed and natural proteins are reported separately, never pooled.** Protenix-v2 single-seq
  scores 0.835 on designs and 0.265 on natural monomers; a pooled mean over a set that is 75 %
  designed (legacy-554) mostly reports how well a model folds idealised backbones.
* **Baseline comparisons need proteins that postdate the baselines' training cutoffs.** The
  FoldBench sets satisfy this by construction (0 of 217 eval-test units predate Protenix-v2's
  2021-09-30 cutoff). Half of legacy-554's designs do not, so a MarinFold-versus-baseline number
  there is contaminated *for the baselines* — compare our own checkpoints to each other and say so.
* **Differences under ~0.005 are ties** ([#204](https://github.com/Open-Athena/MarinFold/issues/204):
  four evaluations of one unchanged checkpoint span 0.0023).

**eval-test has a read budget.** It is a held-out confirmation set; selecting on it destroys it.
Every read gets logged in
[`experiments/exp245_evals_foldbench_held_out_monomers/data/eval_test_reads.md`](https://github.com/Open-Athena/MarinFold/blob/main/experiments/exp245_evals_foldbench_held_out_monomers/data/eval_test_reads.md).

In [ ]:
# @title Scoreboard { display-mode: "form" }
SET_FILTERS = {
    "eval-val": lambda frame: frame.eval_set == "eval-val",
    "eval-test": lambda frame: frame.eval_set == "eval-test",
    "eval-denovo": lambda frame: frame.eval_set == "eval-denovo",
    # eval-val + eval-test: every natural monomer in FoldBench, 314 of them.
    "FoldBench natural monomers": lambda frame: frame.eval_set.isin(["eval-val", "eval-test"]),
    "everything": lambda frame: frame.stem.notna(),
}


def selected_units(frame: pd.DataFrame) -> pd.DataFrame:
    if UNIVERSE == "legacy-554":
        return frame
    return frame[SET_FILTERS[EVAL_SET](frame)]


def bootstrap_mean(values: np.ndarray, draws: int = 2_000, seed: int = 0) -> tuple[float, float, float]:
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(draws, len(values)))].mean(axis=1)
    return float(values.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def scoreboard(units: pd.DataFrame, scores: pd.DataFrame) -> pd.DataFrame:
    rows = []
    scores = scores[(scores.range == RANGE) & (scores.cut == METRIC)]
    for split, subset in (("designed", units[units.designed == 1]),
                          ("natural", units[units.designed == 0])):
        if subset.empty:
            continue
        keys = set(zip(subset.dataset, subset.stem))
        in_split = scores[[key in keys for key in zip(scores.dataset, scores.stem)]]
        for predictor, group in in_split.groupby("predictor"):
            mean, low, high = bootstrap_mean(group.value.values)
            rows.append(dict(split=split, n=len(group), predictor=predictor,
                             value=mean, ci_low=low, ci_high=high))
    return pd.DataFrame(rows).sort_values(["split", "value"], ascending=[True, False])


units = selected_units(targets)
if UNIVERSE == "foldbench-monomers" and EVAL_SET in ("eval-test", "FoldBench natural monomers", "everything"):
    print("!! eval-test is a HELD-OUT set with a read budget. If this read informs a decision or a\n"
          "   published claim, append a row to exp245's data/eval_test_reads.md saying why.\n")

board = scoreboard(units, published)
print(f"{UNIVERSE} · {EVAL_SET if UNIVERSE == 'foldbench-monomers' else 'all 554'} · "
      f"{METRIC} ({RANGE}-range)\n")
print(board.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

for split in board.split.unique():
    panel = board[board.split == split].reset_index(drop=True)
    if FOCUS not in set(panel.predictor):
        continue
    rank = int(panel.index[panel.predictor == FOCUS][0]) + 1
    row = panel[panel.predictor == FOCUS].iloc[0]
    print(f"\n{FOCUS} on {split}: {row.value:.3f} [{row.ci_low:.3f}, {row.ci_high:.3f}], "
          f"rank {rank} of {len(panel)}")

splits = [split for split in ("natural", "designed") if (board.split == split).any()]
fig, axes = plt.subplots(1, len(splits), figsize=(7.2 * len(splits), 4.4), squeeze=False)
for axis, split in zip(axes[0], splits):
    panel = board[board.split == split].sort_values("value")
    colors = ["#C44E52" if predictor == FOCUS else
              "#DD8452" if predictor.startswith(("#", "MarinFold")) else "#4C72B0"
              for predictor in panel.predictor]
    axis.barh(panel.predictor, panel.value, color=colors,
              xerr=[panel.value - panel.ci_low, panel.ci_high - panel.value],
              error_kw=dict(ecolor="0.3", lw=1.2, capsize=3))
    axis.set(xlabel=f"{METRIC} ({RANGE}-range)", xlim=(0, 1),
             title=f"{split} · n={int(panel.n.iloc[0])}")
    axis.grid(axis="x", alpha=0.3)
fig.suptitle(f"{UNIVERSE}{' · ' + EVAL_SET if UNIVERSE == 'foldbench-monomers' else ''}"
             f"  —  red = {FOCUS}, orange = our other checkpoints", y=1.02)
fig.tight_layout()
plt.show()

## 2. Protein browser

One row per protein: every predictor's score side by side, joined to
[#247](https://github.com/Open-Athena/MarinFold/issues/247)'s per-protein features (contact order,
secondary-structure content, MSA depth, best identity to the training corpus, annotations…).

Use it to pick a protein for parts 3–4 — the interesting ones are usually the extremes of
`delta` (`FOCUS` minus `BASELINE`), not the top of the list.

The second cell pairs the two predictors protein by protein and breaks the difference out by
homology stratum, viral status and designed status. Set `FOCUS` to a decontaminated
[#232](https://github.com/Open-Athena/MarinFold/issues/232) checkpoint and `BASELINE` to the
contaminated `#199 cooldown` and it answers the question those two checkpoints exist to answer:
how much of the cooldown's lead is leakage.

In [ ]:
# @title Per-protein table + scatter { display-mode: "form" }
available = sorted(published.predictor.unique())
missing = [name for name in (FOCUS, BASELINE) if name not in available]
if missing:
    raise ValueError(f"{missing} were not scored on {UNIVERSE}. Predictors here: {available}")

wide = (published[(published.range == RANGE) & (published.cut == METRIC)]
        .pivot_table(index=["dataset", "stem"], columns="predictor", values="value")
        .reset_index())
columns = ["dataset", "stem", "L", "eval_set", "designed", "is_viral",
           "exp199_best_identity", "exp199_stratum", "title"]
browser = (units[[c for c in columns if c in units.columns]]
           .merge(wide, on=["dataset", "stem"], how="inner", validate="one_to_one"))
browser["delta"] = browser[FOCUS] - browser[BASELINE]
feature_columns = ["stem", "relative_contact_order", "frac_helix", "frac_sheet",
                   "msa_log_depth", "knn_best_identity", "n_pfam"]
browser = browser.merge(features[[c for c in feature_columns if c in features.columns]],
                        on="stem", how="left", validate="many_to_one")
browser = browser.sort_values("delta")

pd.set_option("display.width", 250, "display.max_columns", 60)
# The contaminated reference rides along when it exists, so the decontamination contrast is
# visible per protein rather than only in the aggregate.
contrast = [name for name in ("#199 cooldown (contaminated)", "MarinFold #199 cooldown")
            if name in browser.columns and name not in (FOCUS, BASELINE)]
show = [c for c in ["dataset", "stem", "L", "eval_set", "designed", "is_viral", FOCUS, BASELINE,
                    "delta", *contrast, "relative_contact_order", "frac_sheet", "msa_log_depth",
                    "exp199_best_identity"] if c in browser.columns]
print(f"--- worst 12 for {FOCUS} vs {BASELINE} ---")
print(browser[show].head(12).to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\n--- best 12 ---")
print(browser[show].tail(12).iloc[::-1].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

if "delta" in browser:
    fig, axis = plt.subplots(figsize=(6.4, 6.0))
    for label, subset, color in (("natural", browser[browser.designed == 0], "#4C72B0"),
                                 ("designed", browser[browser.designed == 1], "#DD8452")):
        axis.scatter(subset[BASELINE], subset[FOCUS], s=26, alpha=0.75, label=label, color=color)
    axis.plot([0, 1], [0, 1], color="0.4", lw=1, ls="--")
    axis.set(xlabel=f"{BASELINE}  ·  {METRIC} ({RANGE})", ylabel=f"{FOCUS}  ·  {METRIC} ({RANGE})",
             xlim=(0, 1.02), ylim=(0, 1.02),
             title=f"{UNIVERSE} · {EVAL_SET if UNIVERSE == 'foldbench-monomers' else 'legacy 554'}"
                   f"\nabove the line = {FOCUS} wins")
    axis.legend(); axis.grid(alpha=0.3)
    plt.show()

In [ ]:
# @title Paired contrast: FOCUS vs BASELINE, and where the difference sits { display-mode: "form" }
# A mean-of-means hides which proteins carry a difference. This pairs the two predictors on the
# same proteins, bootstraps the paired delta, and breaks it out by the annotations #245 carries —
# homology to #199's (un-decontaminated) training set, viral status, designed status. On a
# decontaminated-vs-contaminated pair those strata are the whole question: a lead that is really
# leakage should be largest where the contaminated model had a close training relative and should
# vanish where it had none.
paired = browser[[FOCUS, BASELINE]].dropna()
delta = (paired[FOCUS] - paired[BASELINE]).values
mean, low, high = bootstrap_mean(delta)
print(f"{FOCUS}  −  {BASELINE}   on {len(paired)} shared proteins")
print(f"  paired delta {mean:+.3f}  [{low:+.3f}, {high:+.3f}]   "
      f"{FOCUS} ahead on {100 * (delta > 0).mean():.0f}% of them")

for column, label in (("exp199_stratum", "identity to #199's training set"),
                      ("is_viral", "viral"), ("designed", "designed")):
    if column not in browser.columns or browser[column].isna().all():
        continue
    frame = browser.dropna(subset=[FOCUS, BASELINE]).copy()
    frame["group"] = frame[column].fillna("no_homolog")
    frame["_delta"] = frame[FOCUS] - frame[BASELINE]
    table = frame.groupby("group").agg(
        n=("_delta", "size"), focus=(FOCUS, "mean"), baseline=(BASELINE, "mean"),
        delta=("_delta", "mean")).rename(columns={"focus": FOCUS, "baseline": BASELINE})
    if len(table) > 1:
        print(f"\n  by {label}:")
        print(table.to_string(float_format=lambda v: f"{v:.3f}"))

# Rank correlation between the delta and raw identity, over the proteins that have a homolog at
# all. Stratum means and this can disagree — the strata differ in difficulty as well as identity.
identity = browser.dropna(subset=[FOCUS, BASELINE, "exp199_best_identity"])
if len(identity) > 10:
    ranked = (identity[FOCUS] - identity[BASELINE]).rank()
    rho = float(np.corrcoef(ranked, identity.exp199_best_identity.rank())[0, 1])
    print(f"\n  spearman(delta, identity to #199 training set) = {rho:+.3f} over {len(identity)} "
          f"proteins with a homolog")

## 3. Predict a contact map

This runs the model. The recipe is the one [#82](https://github.com/Open-Athena/MarinFold/issues/82)
settled on and every published number since uses — **rollout + resample**:

1. Build `N_ROLLOUTS` *different* realizations of the same protein's document. contacts-v1 documents
   carry two nuisance randomizations (where the N-terminus lands in position-token space, and the
   order of the sequence statements), so each realization is a different prompt for the same protein.
2. Sample a contact-section completion from each (`temperature 1.0, top-p 0.95`, no top-k), with a
   generous token budget of `min(8192 − prompt, 6·L + 128)` so truncation is never the reason a
   document is short.
3. Count, for every residue pair, how many rollouts asserted it. That vote count is the score.
   Ties (mostly the large zero-vote mass) are broken by the pairwise log-probability readout.

Do not substitute the older pairwise-only readout for this: identical weights score ~0.086 lower
in R-precision under it, which reads like two generations of model progress.

**Metrics** use #89's implementation — candidate pairs are those between two *resolved* residues at
separation ≥ 6, a pair is a true contact at pyconfind degree ≥ 0.001, and R-precision is precision
in the top-R ranked pairs where R is that protein's true-contact count.

In [ ]:
# @title Rollout + metric implementation { display-mode: "form" }
from sklearn.metrics import roc_auc_score

from marinfold.document_structures.contacts_v1 import (
    InferenceConfig, predict, structure_from_sequence)

MIN_DEGREE, MIN_SEPARATION = 0.001, 6


def resolve_backend() -> str:
    """`vllm` where it is installed and supported, `transformers` otherwise."""
    import torch

    if BACKEND == "transformers":
        return "transformers"
    usable = (importlib.util.find_spec("vllm") is not None and torch.cuda.is_available()
              and torch.cuda.get_device_capability()[0] >= 8)
    if BACKEND == "vllm" and not usable:
        print("note: vLLM was asked for but is not usable here — using transformers")
    return "vllm" if usable else "transformers"


def gpu_plan(model: str, length: int, backend: str = "transformers") -> tuple[str, int]:
    """Pick a dtype and a rollout batch size the current GPU can actually hold.

    Two things go wrong on a small or older card, both quietly:

    * **dtype.** These checkpoints were trained in bfloat16, but bfloat16 has no tensor-core
      support before compute capability 8.0 — on Colab's free T4 (7.5) it runs, slowly, through
      emulation. float16 uses the hardware and is what this picks there. `marinfold` never
      downgrades a dtype behind your back, so the choice is made here and printed.
    * **KV cache.** The rollouts for one protein decode as a batch, and the cache is
      `2 x layers x kv_heads x head_dim x dtype_bytes` per token per row — 48 KB for this
      architecture (24 layers, 8 KV heads, head dim 64), so a 1,500-residue protein at 32 rows
      wants ~17 GB of cache alone. A 16 GB card runs out long before a 24 GB one does, so the
      batch is sized from free memory rather than from a constant.

    The number returned is a ceiling. `marinfold`'s transformers backend applies its own
    roughly-constant-cache heuristic on top (about 20,000 residues per batch), so the batch you
    see running can be smaller than the one announced here — on an A100 that is what binds for
    proteins past ~200 residues.

    Returns `(dtype_name, batch_size)`.
    """
    import json
    from pathlib import Path

    import torch

    from marinfold.registry import resolve_model

    if not torch.cuda.is_available():
        # CPU: no KV-cache pressure to model, but a big batch just thrashes.
        return ("float32" if DTYPE == "auto" else DTYPE), (BATCH_SIZE if BATCH_SIZE > 0 else 8)

    major, _ = torch.cuda.get_device_capability()
    dtype = DTYPE if DTYPE != "auto" else ("bfloat16" if major >= 8 else "float16")
    dtype_bytes = 4 if dtype == "float32" else 2

    config = json.loads((Path(resolve_model(model)) / "config.json").read_text())
    per_token = (2 * config["num_hidden_layers"] * config["num_key_value_heads"]
                 * config["head_dim"] * dtype_bytes)
    # One row holds the prompt (~2 tokens/residue + framing) plus the generation budget.
    tokens_per_row = (2 * length + 32) + min(8_192, 6 * length + 128)

    if backend == "vllm":
        # vLLM pages its own KV cache inside its memory share and schedules its own batches, so
        # there is nothing to size here; the batch argument is ignored by that backend.
        free, total = torch.cuda.mem_get_info()
        print(f"gpu: {torch.cuda.get_device_name(0)} (cc {major}.x, {total / 2**30:.0f} GiB, "
              f"{free / 2**30:.1f} GiB free) -> vLLM, dtype {dtype}, "
              f"{GPU_MEMORY_UTILIZATION:.0%} memory share")
        return dtype, N_ROLLOUTS

    ceiling = BATCH_SIZE if BATCH_SIZE > 0 else N_ROLLOUTS
    free, total = torch.cuda.mem_get_info()
    # The weights are not resident yet — predict() loads them after this returns — so subtract
    # them: 1.47B parameters at 2 bytes, plus room for activations and the tie-break pass.
    weights_and_slack = int(3.6e9)
    headroom = max(0.0, (free - weights_and_slack) * 0.7)
    fits = int(headroom // (per_token * tokens_per_row)) if headroom else 0
    batch = max(1, min(ceiling, fits))
    limit = "memory" if fits < ceiling else ("N_ROLLOUTS" if BATCH_SIZE <= 0 else "BATCH_SIZE")
    print(f"gpu: {torch.cuda.get_device_name(0)} (cc {major}.x, {total / 2**30:.0f} GiB, "
          f"{free / 2**30:.1f} GiB free) -> dtype {dtype}, rollout batch {batch} "
          f"(limited by {limit}; {per_token * tokens_per_row / 2**20:.0f} MiB of KV cache per "
          f"rollout at L={length})")
    return dtype, batch


def release_gpu_memory() -> None:
    """Return the last model's memory to the device, so the next one has room."""
    import gc

    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
RANGES = {"all": (6, None), "short": (6, 11), "medium": (12, 23), "long": (24, None)}
CUTS = (("L", lambda L, true: L), ("L/2", lambda L, true: max(1, L // 2)),
        ("L/5", lambda L, true: max(1, L // 5)), ("R", lambda L, true: true))


def true_matrix(length: int, contacts) -> np.ndarray:
    """#89's ground-truth contact matrix: degree >= 0.001, separation >= 6."""
    matrix = np.zeros((length, length), bool)
    for i, j, degree in contacts:
        i, j = int(i), int(j)
        if degree >= MIN_DEGREE and (j - i) >= MIN_SEPARATION and i < j < length:
            matrix[i, j] = matrix[j, i] = True
    return matrix


def candidate_pairs(resolved) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Upper-triangle pairs of resolved residues, and their sequence separations."""
    resolved = np.asarray(resolved)
    left, right = np.triu_indices(len(resolved), k=1)
    i, j = resolved[left], resolved[right]
    return i, j, j - i


def score_metrics(score: np.ndarray, record: dict) -> pd.DataFrame:
    """precision @ {L, L/2, L/5, R} + AUC per separation range — #89's metric_rows."""
    length = record["L"]
    truth = true_matrix(length, record["contacts"])
    i, j, separation = candidate_pairs(record["resolved"])
    pair_scores, pair_truth = score[i, j], truth[i, j].astype(int)
    rows = []
    for name, (low, high) in RANGES.items():
        in_range = separation >= low
        if high is not None:
            in_range &= separation <= high
        values, labels = pair_scores[in_range], pair_truth[in_range]
        n_candidate, n_true = int(values.size), int(labels.sum())
        ranked = labels[np.argsort(-values, kind="mergesort")] if n_candidate else None
        for cut, size_of in CUTS:
            target = int(size_of(length, n_true))
            top = min(target, n_candidate)
            rows.append(dict(range=name, cut=cut, n_candidate=n_candidate, n_true=n_true,
                             value=float(ranked[:top].sum()) / top if top > 0 else np.nan))
        auc = (float(roc_auc_score(labels, values))
               if n_candidate and 0 < n_true < n_candidate else np.nan)
        rows.append(dict(range=name, cut="AUC", n_candidate=n_candidate, n_true=n_true, value=auc))
    return pd.DataFrame(rows)


def resolve_target(name: str, universe: tuple = None) -> pd.Series:
    """Look a protein up by `stem` or by the fully qualified `dataset__stem`."""
    frame = targets if universe is None else universe[0]
    label = UNIVERSE if universe is None else "the given universe"
    if "__" in name:
        dataset, stem = name.split("__", 1)
        match = frame[(frame.dataset == dataset) & (frame.stem == stem)]
    else:
        match = frame[frame.stem == name]
    if match.empty:
        raise ValueError(f"{name!r} is not in {label}. Part 2 lists the {len(frame)} that are.")
    if len(match) > 1:
        # legacy-554 holds 7ur7_A and 8ah9_A twice, under two datasets, with different sequences.
        options = ", ".join(f"{d}__{s}" for d, s in zip(match.dataset, match.stem))
        raise ValueError(f"{name!r} is ambiguous in {label} — ask for one of: {options}")
    return match.iloc[0]


def fold(name: str, model: str, n_rollouts: int = None, universe: tuple = None) -> dict:
    """Run one checkpoint on one protein and score it. Returns score matrix + metrics.

    `universe` is an optional `(targets, ground_truth)` pair, for folding a protein that is not in
    the universe the notebook is currently pointed at — part 5 uses it for Top7, which lives in
    legacy-554.
    """
    target = resolve_target(name, universe)
    stem = target.stem
    backend = resolve_backend()
    dtype, batch_size = gpu_plan(model, int(target.L), backend)
    config = InferenceConfig(
        model=model, backend=backend, method="rollout", keep_matrix=True,
        n_rollouts=n_rollouts or N_ROLLOUTS, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
        batch_size=batch_size, dtype=dtype, min_seq_separation=MIN_SEPARATION,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION)
    record = next(iter(predict(config, structures=[
        structure_from_sequence(target.input_seq, entry_id=stem)])))
    # predict() builds its own backend and drops it on return, but the CUDA allocator holds the
    # freed blocks, so a second checkpoint loaded later in the session adds to the first rather
    # than replacing it. Parts 3, 4 and 5 each load one, which is more than a 16 GB T4 has.
    release_gpu_memory()
    # The band within MIN_SEPARATION comes back as NaN; it is never a candidate pair, but a NaN
    # would poison argsort, so push it below every real score.
    score = np.nan_to_num(np.asarray(record["score_matrix"], dtype=float), nan=-1e9)
    truth = (ground_truth if universe is None else universe[1])[(target.dataset, target.stem)]
    metrics = score_metrics(score, truth)
    return dict(stem=stem, model=model, target=target, score=score, truth=truth, metrics=metrics)


def headline(result: dict) -> str:
    metrics = result["metrics"]
    row = metrics[(metrics.range == "all") & (metrics.cut == "R")].iloc[0]
    auc = metrics[(metrics.range == "all") & (metrics.cut == "AUC")].value.iloc[0]
    long_r = metrics[(metrics.range == "long") & (metrics.cut == "R")].value.iloc[0]
    return (f"R-precision {row.value:.3f} (all) / {long_r:.3f} (long)   AUC {auc:.3f}   "
            f"[L={result['target'].L}, {int(row.n_true)} true contacts of "
            f"{int(row.n_candidate)} candidate pairs]")

In [ ]:
# @title Fold `PROTEIN` with `MODEL` { display-mode: "form" }
import time

start = time.time()
result = fold(PROTEIN, MODEL)
print(f"{MODEL} on {PROTEIN}  ({time.time() - start:.0f}s, {N_ROLLOUTS} rollouts)")
print(headline(result))

# How this compares to the published score for the same protein, where one exists. It will not
# match to the digit: the recipe is stochastic, this runs under transformers rather than the vLLM
# the eval harness uses, and the packaged recipe adds the tie-break. Same ballpark is the check.
target = result["target"]
match = published[(published.dataset == target.dataset) & (published.stem == target.stem)
                  & (published.range == "all") & (published.cut == "R")]
if not match.empty:
    print("\npublished R-precision (all) for this protein:")
    print(match[["predictor", "value"]].sort_values("value", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# @title Contact-map figure { display-mode: "form" }
from matplotlib.colors import LinearSegmentedColormap


def plot_contact_map(result: dict, axes=None, title: str = None):
    """Ground truth against prediction, twice: as a map, and as the top-R call."""
    score, truth_record = result["score"], result["truth"]
    length = truth_record["L"]
    truth = true_matrix(length, truth_record["contacts"])
    resolved = np.zeros(length, bool)
    resolved[np.asarray(truth_record["resolved"])] = True
    candidate = np.outer(resolved, resolved)
    candidate &= np.abs(np.subtract.outer(np.arange(length), np.arange(length))) >= MIN_SEPARATION

    # Panel 1: prediction above the diagonal, ground truth below it. Pairs the metric cannot
    # see — either endpoint unresolved in the deposited structure, or separation < 6 — are blanked
    # in both triangles, so the picture is exactly what the score is computed over.
    votes = np.where(candidate, score, np.nan)
    upper = np.triu(np.ones_like(votes, bool), k=1)
    panel = np.where(upper, votes, np.nan)
    panel_truth = np.where(~upper & truth, 1.0, np.nan)

    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=(13.4, 6.4))
    heat = LinearSegmentedColormap.from_list("votes", ["#F2F2F2", "#F4C36B", "#C44E52", "#3B0A0C"])
    image = axes[0].imshow(panel, cmap=heat, origin="lower", interpolation="none",
                           vmin=0, vmax=max(1.0, np.nanmax(panel)))
    axes[0].imshow(panel_truth, cmap=LinearSegmentedColormap.from_list("gt", ["#333333", "#333333"]),
                   origin="lower", interpolation="none", vmin=0, vmax=1)
    axes[0].plot([0, length - 1], [0, length - 1], color="0.6", lw=0.8)
    plt.colorbar(image, ax=axes[0], fraction=0.046,
                 label=f"rollout votes (of {N_ROLLOUTS}, + tie-break)")
    axes[0].set(xlabel="residue j", ylabel="residue i",
                title=(title or f"{result['stem']} · {result['model']}") +
                      "\nupper: model votes   ·   lower: ground truth")

    # Panel 2: the top-R ranked pairs, right and wrong, over the ground truth.
    metrics = result["metrics"]
    n_true = int(metrics[(metrics.range == "all") & (metrics.cut == "R")].n_true.iloc[0])
    ranked = np.argsort(-np.where(candidate & upper, score, -np.inf), axis=None, kind="mergesort")
    top = np.unravel_index(ranked[:n_true], score.shape)
    hit = truth[top]
    axes[1].imshow(np.where(truth, 1.0, np.nan), cmap=LinearSegmentedColormap.from_list(
        "gt", ["#D8D8D8", "#D8D8D8"]), origin="lower", interpolation="none", vmin=0, vmax=1)
    for mask, color, label in ((hit, "#2A7F43", "correct"), (~hit, "#C44E52", "wrong")):
        axes[1].scatter(np.asarray(top[1])[mask], np.asarray(top[0])[mask], s=9, color=color,
                        label=f"{label} ({int(mask.sum())})")
        axes[1].scatter(np.asarray(top[0])[mask], np.asarray(top[1])[mask], s=9, color=color)
    axes[1].plot([0, length - 1], [0, length - 1], color="0.6", lw=0.8)
    axes[1].set(xlim=(-0.5, length - 0.5), ylim=(-0.5, length - 0.5),
                xlabel="residue j", ylabel="residue i",
                title=f"top-{n_true} predicted contacts over ground truth (grey)\n{headline(result)}")
    axes[1].legend(loc="lower right", fontsize=9)
    return axes


plot_contact_map(result)
plt.tight_layout()
plt.show()

## 4. Two checkpoints on the same protein

The same protein, folded twice. Useful for reading what a training change actually bought — the
aggregate says "+0.03 R-precision", the maps say whether it found a different fold or sharpened the
same one.

The default pair is the **contamination contrast**: `MODEL` is the current default checkpoint,
whose corpora were never filtered against FoldBench, and `MODEL_B` is
[#232](https://github.com/Open-Athena/MarinFold/issues/232)'s `m2-p06` — the same recipe trained
from scratch on [#225](https://github.com/Open-Athena/MarinFold/issues/225)'s corpora, with every
sequence matching any eval protein at ≥30 % identity over ≥50 % of the shorter one removed. It is
the better of the two finals [#244](https://github.com/Open-Athena/MarinFold/pull/244) selected
(0.5916 vs 0.5789 on the legacy 554) and the checkpoint the
[#213](https://github.com/Open-Athena/MarinFold/issues/213) leakage objection does not apply to.

The aggregate gap between them is 0.039 on legacy-554 and 0.069 on eval-val, and part 2 will show
you which proteins carry it. What the maps add is *how* it is carried — a fold found by one and
missed by the other, or the same fold with sharper votes.

For the corpus-growth story instead, set `MODEL_B` to `contacts-v1-exp117-1.5B` (AFDB alone,
before the ESM-Atlas half was added).

This cell downloads a second checkpoint the first time it runs — about 5.9 GB for `m2-p06`, which
is stored float32 and cast at load; budget a few minutes on top of the fold itself.

In [ ]:
# @title Fold with both checkpoints { display-mode: "form" }
result_b = fold(PROTEIN, MODEL_B)
print(f"{MODEL:<38} {headline(result)}")
print(f"{MODEL_B:<38} {headline(result_b)}")

figure, axes = plt.subplots(2, 2, figsize=(13.4, 12.4))
plot_contact_map(result, axes=axes[0], title=f"{PROTEIN} · {MODEL}")
plot_contact_map(result_b, axes=axes[1], title=f"{PROTEIN} · {MODEL_B}")
figure.tight_layout()
plt.show()

comparison = (result["metrics"].merge(result_b["metrics"], on=["range", "cut"],
                                      suffixes=("_a", "_b"))
              .assign(delta=lambda frame: frame.value_a - frame.value_b))
print(f"\n{MODEL} (a) vs {MODEL_B} (b)")
print(comparison[comparison.cut.isin(["R", "AUC"])][["range", "cut", "value_a", "value_b", "delta"]]
      .to_string(index=False, float_format=lambda v: f"{v:+.3f}"))

## 5. Manuscript figures

Renders the data panels a write-up needs, as standalone files in `figures/` — PNG at 300 dpi and
PDF (vector) — with no titles and no panel letters baked in, so they can be captioned and lettered
wherever they land.

| file | what it is | needs |
|---|---|---|
| `contacts_v1_document_top7.txt` | the real contacts-v1 document for Top7, for drawing the format panel | CPU |
| `contact_map_top7.*` | Top7's map, prediction above the diagonal and observed contacts below | **GPU** |
| `contact_map_top7_side_by_side.*` | the same thing as two panels, observed next to predicted | **GPU** |
| `rprecision_natural.*` / `rprecision_designed.*` | contact-prediction R-precision, both protein classes | CPU |
| `gdt_ts_natural.*` / `gdt_ts_designed.*` | Helico GDT-TS, both protein classes | CPU |

**The protein sets, and why these ones.** Natural = the 314 FoldBench natural monomers
(eval-val + eval-test). Designed = FoldBench's 19 de novo monomers (`eval-denovo`). That is the
only designed set where a baseline comparison is legitimate: exp65's 396 designs are 20x larger
and already scored, but **50.5 % of them were deposited on or before Protenix-v2's 2021-09-30
cutoff**, so on that set the baselines are the contaminated party. 19 is small — the intervals say
how small — but it is honest, and it is the same set the Helico GDT-TS panels use, so both figures
describe the same proteins.

**Which MarinFold checkpoint.** Both are drawn: `#232 m2-p06`, trained on corpora filtered against
all of FoldBench, and the contaminated `#199 cooldown`. Helico's contact arm was built from
m2-p06, so m2-p06 is the one that makes the contact and structure figures consistent; the cooldown
is drawn beside it because dropping it would hide that our best number comes from a model whose
training data was never filtered against the eval set. Drop either from `FIGURE_PREDICTORS` if the
manuscript has room for only one.

In [ ]:
# @title Figure setup — style, output dir, and the Helico results { display-mode: "form" }
FIGURE_DIR = "figures"  # @param {type:"string"}
FIGURE_DPI = 300  # @param {type:"integer"}
from pathlib import Path

from matplotlib.colors import LinearSegmentedColormap

FIGURE_DIR = Path(FIGURE_DIR)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Contact predictors to draw, with the names the manuscript should use.
FIGURE_PREDICTORS = [
    ("Protenix-v2 + MSA", "Protenix-v2 + MSA"),
    ("ESMFold2", "ESMFold2"),
    ("ESMFold", "ESMFold"),
    ("#199 cooldown (contaminated)", "MarinFold"),
    ("#232 m2-p06 (decontaminated)", "MarinFold (decontaminated)"),
    ("Protenix-v2 single-seq", "Protenix-v2, single sequence"),
]
# Helico structure arms, same idea. `off` and `oracle` bracket what contact conditioning can do.
HELICO_ARMS = [
    ("protenix_v2_msa", "Protenix-v2 + MSA"),
    ("esmfold2", "ESMFold2"),
    ("oracle", "Helico + true contacts"),
    ("mf_L", "Helico + MarinFold contacts"),
    ("protenix_v2_single_seq", "Protenix-v2, single sequence"),
    ("off", "Helico, no contacts"),
]
HELICO = ("https://huggingface.co/buckets/timodonnell/helico-experiments/resolve/"
          "exp14_foldbench_held_out_monomers")

plt.rcParams.update({
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 140, "savefig.bbox": "tight", "pdf.fonttype": 42,
})
MARINFOLD_COLOR, OTHER_COLOR = "#C44E52", "#7A8DA6"


def save_figure(figure, name):
    """Write one panel as PNG (300 dpi) and PDF — no title, no panel letter."""
    for suffix in ("png", "pdf"):
        figure.savefig(FIGURE_DIR / f"{name}.{suffix}", dpi=FIGURE_DPI)
    print(f"wrote {FIGURE_DIR / name}.png and .pdf")


def helico_per_target():
    """Helico's per-protein structure metrics over exp245's units (public, anonymous read).

    Restricted to targets every arm scored, so the arms are compared on one protein set rather
    than on whichever ones each of them happened to finish.
    """
    frame = pd.read_csv(io.BytesIO(fetch(f"{HELICO}/scores/per_target.csv")))
    scored = frame[frame.status == "ok"]
    complete = scored.groupby("target_id").arm.nunique()
    keep = set(complete[complete == scored.arm.nunique()].index)
    dropped = scored.target_id.nunique() - len(keep)
    if dropped:
        print(f"note: {dropped} targets are missing from at least one Helico arm and are excluded")
    return scored[scored.target_id.isin(keep)]


legacy = load_universe("legacy-554")
figure_target = resolve_target(FIGURE_PROTEIN, legacy)
figure_truth = legacy[1][(figure_target.dataset, figure_target.stem)]
print(f"figure protein: {figure_target.dataset}__{figure_target.stem}  L={figure_target.L}  "
      f"{len(figure_truth['contacts'])} ground-truth contacts")

In [ ]:
# @title The contacts-v1 document for Top7 (source for the format panel) { display-mode: "form" }
# Not a figure — the actual document text, so a format panel can be drawn from the real thing
# rather than a paraphrase. The full document goes to the file; a few lines are printed here.
from marinfold.document_structures.contacts_v1 import (
    GenerationConfig, RawContact, build_document, residues_from_sequence)

document = build_document(
    figure_target.stem,
    residues_from_sequence(figure_target.input_seq),
    [RawContact(seq_i=int(i), seq_j=int(j), degree=float(degree))
     for i, j, degree in figure_truth["contacts"]],
    config=GenerationConfig(),
)
path = FIGURE_DIR / "contacts_v1_document_top7.txt"
path.write_text(document.document)
statements = document.document.count("<contact>")
print(f"wrote {path}  ({figure_target.stem}, L={figure_target.L}, {statements} contact statements)")
print()
head, _, tail = document.document.partition("<begin_statements>")
print(" ".join(head.split()[:12]), "...")
print("<begin_statements>")
for statement in tail.split("<contact>")[1:5]:
    print("<contact> " + " ".join(statement.split()[:2]))
print("...")

In [ ]:
# @title Contact map for Top7 (GPU) { display-mode: "form" }
# One square panel: the model's prediction above the diagonal, the deposited structure's contacts
# below it. Pairs the metric cannot see (unresolved residue, or separation < 6) are left blank.
figure_result = fold(FIGURE_PROTEIN, FIGURE_MODEL, universe=legacy)
print(f"{FIGURE_MODEL} on {figure_target.stem}: {headline(figure_result)}")

score, length = figure_result["score"], figure_truth["L"]
truth = true_matrix(length, figure_truth["contacts"])
resolved = np.zeros(length, bool)
resolved[np.asarray(figure_truth["resolved"])] = True
candidate = np.outer(resolved, resolved)
candidate &= np.abs(np.subtract.outer(np.arange(length), np.arange(length))) >= MIN_SEPARATION
upper = np.triu(np.ones((length, length), bool), k=1)

heat = LinearSegmentedColormap.from_list("votes", ["#FFFFFF", "#F4C36B", "#C44E52", "#3B0A0C"])
figure, axis = plt.subplots(figsize=(3.6, 3.6))
votes = np.where(candidate & upper, score, np.nan)
image = axis.imshow(votes / max(1.0, np.nanmax(votes)), cmap=heat, origin="lower",
                    interpolation="none", vmin=0, vmax=1)
axis.imshow(np.where(~upper & truth, 1.0, np.nan), origin="lower", interpolation="none",
            cmap=LinearSegmentedColormap.from_list("gt", ["#2F2F2F", "#2F2F2F"]), vmin=0, vmax=1)
axis.plot([0, length - 1], [0, length - 1], color="0.75", lw=0.6)
axis.set(xlabel="residue", ylabel="residue")
axis.set_xticks([0, length // 2, length - 1])
axis.set_yticks([0, length // 2, length - 1])
for spine in ("top", "right"):
    axis.spines[spine].set_visible(True)
bar = figure.colorbar(image, ax=axis, fraction=0.046, pad=0.03, ticks=[0, 0.5, 1.0])
bar.set_label("predicted contact confidence", fontsize=8)
bar.ax.set_yticklabels(["0", "", "1"])
axis.text(0.03, 0.95, "predicted", transform=axis.transAxes, ha="left", va="top", fontsize=8)
axis.text(0.97, 0.05, "observed", transform=axis.transAxes, ha="right", va="bottom", fontsize=8)
save_figure(figure, "contact_map_top7")
plt.show()

In [ ]:
# @title Contact map for Top7, side by side (GPU — reuses the fold above) { display-mode: "form" }
# The same prediction as the mirrored panel above, drawn instead as two square panels — observed
# on the left, predicted on the right — for a figure where the reader should not have to reflect
# one triangle onto the other to compare them. Both panels are masked to the pairs the metric
# scores, so a blank region means "not scorable", not "no contact".
observed = np.where(candidate & truth, 1.0, np.nan)
predicted = np.where(candidate, score, np.nan)
predicted = predicted / max(1.0, np.nanmax(predicted))

figure, axes = plt.subplots(1, 2, figsize=(6.9, 3.5), sharey=True)
axes[0].imshow(observed, origin="lower", interpolation="none", vmin=0, vmax=1,
               cmap=LinearSegmentedColormap.from_list("gt", ["#2F2F2F", "#2F2F2F"]))
axes[0].set(xlabel="residue", ylabel="residue")
image = axes[1].imshow(predicted, origin="lower", interpolation="none", vmin=0, vmax=1, cmap=heat)
axes[1].set(xlabel="residue")
for axis, note in zip(axes, ("observed", "predicted")):
    axis.plot([0, length - 1], [0, length - 1], color="0.8", lw=0.6)
    axis.set_xticks([0, length // 2, length - 1])
    axis.set_yticks([0, length // 2, length - 1])
    axis.text(0.03, 0.95, note, transform=axis.transAxes, ha="left", va="top", fontsize=8)
    for spine in ("top", "right"):
        axis.spines[spine].set_visible(True)
bar = figure.colorbar(image, ax=axes[1], fraction=0.046, pad=0.03, ticks=[0, 0.5, 1.0])
bar.set_label("predicted contact confidence", fontsize=8)
bar.ax.set_yticklabels(["0", "", "1"])
save_figure(figure, "contact_map_top7_side_by_side")
plt.show()

In [ ]:
# @title R-precision panels (CPU) { display-mode: "form" }
# Two panels, one per protein class, from #245's published per-protein scores. Natural is
# eval-val + eval-test (314); designed is eval-denovo (19), the only designed set that postdates
# the baselines' training cutoffs.
if UNIVERSE != "foldbench-monomers":
    raise SystemExit("set UNIVERSE to 'foldbench-monomers' above — these panels are that set")

FIGURE_SETS = {
    "natural": lambda frame: frame.eval_set.isin(["eval-val", "eval-test"]),
    "designed": lambda frame: frame.eval_set == "eval-denovo",
}


def rprecision_panel(selector):
    """Mean R-precision per predictor over one protein class, with a bootstrap interval."""
    units = targets[selector(targets)]
    keys = set(zip(units.dataset, units.stem))
    scores = published[(published.range == "all") & (published.cut == "R")]
    scores = scores[[key in keys for key in zip(scores.dataset, scores.stem)]]
    rows = []
    for predictor, label in FIGURE_PREDICTORS:
        values = scores[scores.predictor == predictor].value.values
        if not len(values):
            continue
        mean, low, high = bootstrap_mean(values)
        rows.append(dict(label=label, predictor=predictor, n=len(values),
                         value=mean, ci_low=low, ci_high=high))
    return pd.DataFrame(rows)


def horizontal_panel(panel, xlabel, highlight, name):
    """One bar-per-predictor panel, sorted, with intervals and value labels."""
    panel = panel.sort_values("value")
    figure, axis = plt.subplots(figsize=(4.4, 0.34 * len(panel) + 0.9))
    colors = [MARINFOLD_COLOR if highlight(row) else OTHER_COLOR
              for row in panel.itertuples()]
    axis.barh(panel.label, panel.value, color=colors, height=0.68,
              xerr=[panel.value - panel.ci_low, panel.ci_high - panel.value],
              error_kw=dict(ecolor="0.25", lw=0.9, capsize=2.5))
    # Past the interval, not past the bar — otherwise the label sits on the error cap.
    for y, row in enumerate(panel.itertuples()):
        axis.text(row.ci_high + 0.02, y, f"{row.value:.2f}", va="center", fontsize=7.5,
                  color="0.25")
    axis.set(xlabel=xlabel, xlim=(0, 1.06))
    axis.grid(axis="x", alpha=0.25, lw=0.6)
    axis.set_axisbelow(True)
    save_figure(figure, name)
    print(panel[["label", "n", "value", "ci_low", "ci_high"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    plt.show()


for name, selector in FIGURE_SETS.items():
    print(f"--- {name} ---")
    horizontal_panel(rprecision_panel(selector), "R-precision",
                     lambda row: row.predictor.startswith("#"), f"rprecision_{name}")

In [ ]:
# @title GDT-TS panels (CPU) { display-mode: "form" }
# Helico folds each protein from a contact map; the arms differ only in where the contacts came
# from, and `Helico, no contacts` / `Helico + true contacts` bracket what conditioning can do.
# Published by Open-Athena/helico's exp14 and read anonymously.
helico = helico_per_target()

for name, mask in (("natural", helico.designed == 0), ("designed", helico.designed == 1)):
    subset = helico[mask]
    rows = []
    for arm, label in HELICO_ARMS:
        values = subset[subset.arm == arm].gdt_ts.values
        if not len(values):
            continue
        mean, low, high = bootstrap_mean(values)
        rows.append(dict(label=label, arm=arm, n=len(values),
                         value=mean, ci_low=low, ci_high=high))
    print(f"--- {name} ---")
    horizontal_panel(pd.DataFrame(rows), "GDT-TS",
                     lambda row: row.arm == "mf_L", f"gdt_ts_{name}")

### Read the designed panels before writing their captions

The designed panels do not say what the natural ones say, and the difference is not subtle. On
natural monomers Helico folds far better from MarinFold's contacts than from none at all. On the
19 designed monomers it folds **worse** with them than without — `Helico + MarinFold contacts`
sits below `Helico, no contacts` — and Protenix-v2 in single-sequence mode is already near the
top of both designed panels. Designed backbones are idealised and single-sequence predictors
handle them well unaided, so imperfect contacts subtract there rather than add.

Any claim the manuscript makes about designed proteins has to survive that panel.

## What this notebook is not

It reads published artifacts and re-runs a recipe; it does not produce eval numbers of record.
Anything worth citing goes through
[exp245's harness](https://github.com/Open-Athena/MarinFold/tree/main/experiments/exp245_evals_foldbench_held_out_monomers)
and gets filed on the issue, for three reasons:

* **The recipe here is not bit-identical to the harness.** It runs under transformers rather than
  vLLM, and the packaged rollout adds a pairwise tie-break the eval worker does not apply. Both
  move a per-protein score by more than the aggregate noise floor.
* **Per-protein scores are noisy.** #204's four evaluations of one unchanged checkpoint agree to
  0.0023 *in aggregate over 554 proteins*; a single protein at 100 rollouts moves much more than
  that between runs.
* **A score is only as good as the set it is on.** A number read off one protein you chose after
  looking at the table is a selected number, and the eval-test read budget exists precisely because
  selection is invisible after the fact.

Two neighbours, so you pick the right notebook:
[`inference_example1.ipynb`](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/inference_example1.ipynb)
runs a checkpoint on **any RCSB entry** (its own ground truth, vLLM or transformers, no eval set),
and
[`fold_from_contacts1.ipynb`](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/fold_from_contacts1.ipynb)
turns predicted contacts into a **3D backbone**. This one is the one anchored to the eval universes
and the published per-protein scores.